# Stage 3: Direct Preference Optimization (DPO)
This notebook implements Direct Preference Optimization (DPO) on top of the SFT model, utilizing preference pairs (prompt, chosen, rejected) to fine-tune the output tone, accuracy, and safety constraints.

In [1]:
# Install libraries in Google Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft transformers accelerate bitsandbytes
!pip install unsloth_zoo
!pip install datasets
!pip install trl


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-spnpc_6y/unsloth_6f6f4588bc9944839ccaffa62a50aee7
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-spnpc_6y/unsloth_6f6f4588bc9944839ccaffa62a50aee7
  Resolved https://github.com/unslothai/unsloth.git to commit 9fa6fd40e1a227a6c77b7e64d33dcfe3d5f617cd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 142.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.7 MB/s eta 0:00:00
   

In [2]:
from google.colab import userdata
from huggingface_hub import login, upload_folder, create_repo
import os

# 2. Push that local merged folder directly to Hugging Face
# Ensure your Hugging Face token is stored as a Colab secret named 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')

# Now you can use hf_token in your login function or other operations
login(token=hf_token)
print("Logged in to Hugging Face successfully!")

Logged in to Hugging Face successfully!


In [3]:
import torch
from unsloth import FastLanguageModel


max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load SFT model for preference alignment
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Bhargav1/qwen2.5-7b-stage2-merged",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,

)

# Apply LoRA for DPO
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
from datasets import load_dataset

# Load the preference dataset (upload 'preference_dataset.jsonl' via the sidebar)
#dataset = load_dataset("json", data_files="preference_dataset.jsonl", split="train")
dataset = load_dataset("json", data_files="/content/drive/MyDrive/AIML-2026/preference_dataset.jsonl", split="train")


# Format inputs for DPO
prompt_format = """Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

def format_dpo(examples):
    formatted = {
        "prompt": [],
        "chosen": [],
        "rejected": []
    }
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        formatted["prompt"].append(prompt_format.format(instruction=prompt))
        formatted["chosen"].append(chosen + tokenizer.eos_token)
        formatted["rejected"].append(rejected + tokenizer.eos_token)
    return formatted

dataset = dataset.map(format_dpo, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [8]:
from trl import DPOTrainer
from transformers import TrainingArguments

# Initialize TrainingArguments
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    max_steps = 200,
    learning_rate = 5e-6,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.0,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = "dpo_outputs",
    remove_unused_columns = False,
)

# Manually add attributes to TrainingArguments for Unsloth's DPOTrainer
training_args.model_init_kwargs = None
training_args.ref_model_init_kwargs = None
training_args.padding_value = tokenizer.pad_token_id
training_args.generate_during_eval = False
training_args.model_adapter_name = None
training_args.ref_adapter_name = None
training_args.reference_free = False
training_args.disable_dropout = False
training_args.use_liger_loss = False
training_args.label_pad_token_id = tokenizer.pad_token_id
training_args.max_prompt_length = 512
training_args.max_completion_length = 512
training_args.max_length = 1024
training_args.truncation_mode = "longest_first"
training_args.precompute_ref_log_probs = False
training_args.use_logits_to_keep = False
training_args.padding_free = False
training_args.beta = 0.1
training_args.label_smoothing = 0.0
training_args.loss_type = "sigmoid"
training_args.loss_weights = None
training_args.use_weighting = False
training_args.f_divergence_type = "kl"
training_args.f_alpha_divergence_coef = 0.0
training_args.dataset_num_proc = None
training_args.tools = None
training_args.sync_ref_model = False
training_args.rpo_alpha = None
training_args.ld_alpha = None # Add this line to fix the new error

# Initialize DPOTrainer using Unsloth wrapper configuration
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth automatically handles saving memory by freezing baseline weights
    args = training_args, # Use the modified training_args
    beta = 0.1, # Implicit reward factor for preference comparison
    train_dataset = dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

dpo_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/content/unsloth_compiled_cache/UnslothDPOTrainer.py:1027: UserWarning: The `padding_value` argument is deprecated and will be removed in version 0.26.0. Please use `pad_token` (str) instead.
  warnings.warn(


Extracting prompt in train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55 | Num Epochs = 29 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.692901,-0.000138,-0.000643,0.325000,0.000505,-101.407814,-88.240891,-1.423364,-1.346947
10,0.692334,0.000285,-0.001795,0.600000,0.002080,-95.903694,-86.188408,-1.450674,-1.382026
15,0.687804,0.003634,-0.006839,0.825000,0.010473,-102.112755,-87.788193,-1.422258,-1.479177
20,0.671887,0.016322,-0.026765,1.000000,0.043087,-98.453491,-85.530838,-1.413359,-1.359257
25,0.641011,0.039190,-0.068677,1.000000,0.107867,-100.249916,-89.345131,-1.411139,-1.381667
30,0.589402,0.094966,-0.127748,1.000000,0.222714,-99.842789,-89.515541,-1.490181,-1.416595
35,0.537038,0.122817,-0.223851,1.000000,0.346668,-98.841850,-94.134506,-1.414900,-1.362923
40,0.472649,0.182571,-0.329229,1.000000,0.511799,-97.204643,-91.122940,-1.375459,-1.320257
45,0.413393,0.241469,-0.439006,1.000000,0.680474,-95.228401,-94.826553,-1.490395,-1.474600
50,0.370154,0.255745,-0.551335,1.000000,0.807080,-95.897667,-93.009361,-1.368551,-1.276784


Unsloth: Restored added_tokens_decoder metadata in dpo_outputs/checkpoint-200/tokenizer_config.json.


TrainOutput(global_step=200, training_loss=0.21060007236897946, metrics={'train_runtime': 1581.7075, 'train_samples_per_second': 1.012, 'train_steps_per_second': 0.126, 'total_flos': 0.0, 'train_loss': 0.21060007236897946, 'epoch': 28.571428571428573})

In [9]:
from unsloth import FastLanguageModel

# Inference test on DPO model
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    prompt_format.format(
        instruction = "What is the difference between the Free Plan and the Pro Plan?",
        response = "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 150, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
What is the difference between the Free Plan and the Pro Plan?

### Response:
The Free Plan allows users to upload videos, practiceyourspeech.com runs the default AWS pipeline, and provides standard reports. The Pro Plan adds custom heuristics, advanced profiles, and direct competition tracking to create a fully immersive public speaking rehearsal platform.


In [10]:
from google.colab import userdata
from huggingface_hub import login, upload_folder, create_repo
import os

# 2. Push that local merged folder directly to Hugging Face
# Ensure your Hugging Face token is stored as a Colab secret named 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')

# Now you can use hf_token in your login function or other operations
login(token=hf_token)
print("Logged in to Hugging Face successfully!")

model.push_to_hub_merged(
    "Bhargav1/qwen2.5-7b-final-merged",
    tokenizer,
    save_method="merged_16bit",
    token=hf_token
)

Logged in to Hugging Face successfully!


Unsloth: Restored added_tokens_decoder metadata in Bhargav1/qwen2.5-7b-final-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   3%|2         | 31.9MB / 1.09GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:42<00:00, 70.57s/it]


Unsloth: Merge process complete. Saved to `/content/Bhargav1/qwen2.5-7b-final-merged`
